In [1]:
import pandas as pd
import numpy as np
import os

# 核心配置区
DATA_DIR = 'G:/quant_data/daily_bar/market_data'

print(" 读取本地因子数据 ")
# 载入经过市值中性化后的纯净非流动性因子 (X)
df_factor = pd.read_csv(os.path.join(DATA_DIR, 'factor_illiq_neutral.csv'), index_col='Date', parse_dates=True)
# 载入隔离未来函数的未来1日真实收益率矩阵 (Y)
df_y_target = pd.read_csv(os.path.join(DATA_DIR, 'y_target_matrix.csv'), index_col='Date', parse_dates=True)

# 确保两张表的时间轴和股票池完美对齐
common_dates = df_factor.index.intersection(df_y_target.index)
df_factor = df_factor.loc[common_dates]
df_y_target = df_y_target.loc[common_dates]

print(f"矩阵对齐成功！正在本地测算 {len(common_dates)} 个交易日的横截面 Rank IC...")

# 核心统计：逐日计算横截面(斯皮尔曼秩相关系数)
ic_series = pd.Series(index=common_dates, dtype=float)

for date in common_dates:
    factor_vector = df_factor.loc[date]
    y_vector = df_y_target.loc[date]
    
    # 剔除由于周末、停牌或移位导致的任何空值
    valid_mask = factor_vector.notnull() & y_vector.notnull()
    
    # 10只股的池子，当天至少要有3只股有数才能算相关系数
    if valid_mask.sum() < 3: 
        continue
        
    # 计算 Rank IC：用 spearman 算法，只认排名，不认绝对值，天然抗极值
    rank_ic = factor_vector[valid_mask].corr(y_vector[valid_mask], method='spearman')
    ic_series.loc[date] = rank_ic

# 计算IR IC
mean_ic = ic_series.mean()
std_ic = ic_series.std()

# 计算因子 IR（信息比率）：均值 / 标准差
factor_ir = mean_ic / std_ic if std_ic != 0 else 0

# 计算 IC 胜率 (IC > 0 的天数占比)
ic_win_rate = (ic_series > 0).sum() / ic_series.notnull().sum()

# 计算 t 统计量：检验 IC 是否显著异于 0
n_days = ic_series.notnull().sum()
t_stat = (mean_ic / std_ic) * np.sqrt(n_days) if std_ic != 0 else 0

print("结果如下：")
print(f" 1. Mean Rank IC : {mean_ic*100:.3f}%")
print(f" 2. Std IC : {std_ic*100:.3f}%")
print(f" 3. Factor IR : {factor_ir:.4f}")
print(f" 4. IC Win Rate : {ic_win_rate*100:.2f}%")
print(f" 5. T-Stat : {t_stat:.4f}")

# 结果存档
ic_series.to_csv(os.path.join(DATA_DIR, 'daily_ic_output.csv'))
print("已存入本地")

 读取本地因子数据 
矩阵对齐成功！正在本地测算 104 个交易日的横截面 Rank IC...
结果如下：
 1. Mean Rank IC : -8.066%
 2. Std IC : 32.550%
 3. Factor IR : -0.2478
 4. IC Win Rate : 39.22%
 5. T-Stat : -2.5029
已存入本地


G:\conda\envs\quant\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [2]:
import pandas as pd
import numpy as np
import os

# 核心配置区
DATA_DIR = 'G:/quant_data/daily_bar/market_data'

print(" 读取本地因子数据 ")
# 载入经过市值中性化后的纯净非流动性因子 (X)
df_factor = pd.read_csv(os.path.join(DATA_DIR, 'factor_pvd_clean.csv'), index_col='Date', parse_dates=True)
# 载入隔离未来函数的未来1日真实收益率矩阵 (Y)
df_y_target = pd.read_csv(os.path.join(DATA_DIR, 'y_target_matrix.csv'), index_col='Date', parse_dates=True)

# 确保两张表的时间轴和股票池完美对齐
common_dates = df_factor.index.intersection(df_y_target.index)
df_factor = df_factor.loc[common_dates]
df_y_target = df_y_target.loc[common_dates]

print(f"矩阵对齐成功！正在本地测算 {len(common_dates)} 个交易日的横截面 Rank IC...")

# 核心统计：逐日计算横截面(斯皮尔曼秩相关系数)
ic_series = pd.Series(index=common_dates, dtype=float)

for date in common_dates:
    factor_vector = df_factor.loc[date]
    y_vector = df_y_target.loc[date]
    
    # 剔除由于周末、停牌或移位导致的任何空值
    valid_mask = factor_vector.notnull() & y_vector.notnull()
    
    # 10只股的池子，当天至少要有3只股有数才能算相关系数
    if valid_mask.sum() < 3: 
        continue
        
    # 计算 Rank IC：用 spearman 算法，只认排名，不认绝对值，天然抗极值
    rank_ic = factor_vector[valid_mask].corr(y_vector[valid_mask], method='spearman')
    ic_series.loc[date] = rank_ic

# 计算IR IC
mean_ic = ic_series.mean()
std_ic = ic_series.std()

# 计算因子 IR（信息比率）：均值 / 标准差
factor_ir = mean_ic / std_ic if std_ic != 0 else 0

# 计算 IC 胜率 (IC > 0 的天数占比)
ic_win_rate = (ic_series > 0).sum() / ic_series.notnull().sum()

# 计算 t 统计量：检验 IC 是否显著异于 0
n_days = ic_series.notnull().sum()
t_stat = (mean_ic / std_ic) * np.sqrt(n_days) if std_ic != 0 else 0

print("结果如下：")
print(f" 1. Mean Rank IC : {mean_ic*100:.3f}%")
print(f" 2. Std IC : {std_ic*100:.3f}%")
print(f" 3. Factor IR : {factor_ir:.4f}")
print(f" 4. IC Win Rate : {ic_win_rate*100:.2f}%")
print(f" 5. T-Stat : {t_stat:.4f}")

# 结果存档
ic_series.to_csv(os.path.join(DATA_DIR, 'daily_ic_output.csv'))
print("已存入本地")

 读取本地因子数据 
矩阵对齐成功！正在本地测算 104 个交易日的横截面 Rank IC...
结果如下：
 1. Mean Rank IC : -2.770%
 2. Std IC : 32.526%
 3. Factor IR : -0.0852
 4. IC Win Rate : 46.46%
 5. T-Stat : -0.8475
已存入本地
